# UWCT Tritonia: background-contrast images (v9) x model size (yolo26s / m / l), D-045: E047-E061

**Plan (D-045, declared in docs/EXPERIMENTS.md before any run).** Every run changes one variable from its parent;
everything else is E010's settings (imgsz 640, 100 epochs, patience 30, batch 16, deterministic). Seeds 0 / 1 / 2:

| arm | images | model | runs |
|---|---|---|---|
| baseline | v7 (raw) | yolo26s | E010 / E042 / E043, registered, **not re-run** |
| M | v7 (raw) | yolo26m | E047 / E048 / E049 |
| L | v7 (raw) | yolo26l | E050 / E051 / E052 |
| S-bgc | v9 (bgc1) | yolo26s | E053 / E054 / E055 |
| M-bgc | v9 (bgc1) | yolo26m | E056 / E057 / E058 |
| L-bgc | v9 (bgc1) | yolo26l | E059 / E060 / E061 |

v9 has v7's 150 frames, day split and boxes. Only the images differ: R = original, G = brighter than the day's
median background (x4), B = CLAHE.

- **Selection metric: val point AP** against the MTrackJ crosses (D-033): all 27 val frames, 10 px one to one, each
  run at its own best-val-F1 threshold. Per arm, the **mean over 3 seeds**.
- An arm **wins** only if its mean beats the baseline mean (0.489) by more than the larger of 0.018 and its own seed
  range. Fewer than 3 finished seeds cannot win. The images x size table is descriptive only.
- Ultralytics' P / R / mAP printed during training are against the fixed boxes: **not accuracy** (D-033).
- **No test split anywhere in this notebook** (CLAUDE.md rule 7). Frames are 15 s apart: sparse-frame/timelapse
  analysis, never "video tracking".

**Before the first run (once):** create the private Kaggle dataset **`uwct-tritonia-v9-inputs`** with two folders:
`tritonia_bgc1_v7/` (from `bgc1_outputs.zip`) and `tritonia_boxes_fixed2/` (the same folder as in
`uwct-tritonia-v7-inputs`). Cell 6 checks every file against frozen v9, so an incomplete upload stops there.

**How to run (interactive session; keep the browser tab open). ESTIMATE: about 2-4 h.** A v7 yolo26s run took about
4-5 min on one T4; m and l take longer, and their exact time is UNKNOWN until the smoke runs.
1. Settings: Accelerator **GPU T4 x2**, **Internet on**. Add-ons -> Secrets -> `UWCT_GITHUB_TOKEN` (ticked for this notebook).
2. **Add Input**, both: `anusidhu7/uwct-tritonia-v7-inputs` and `anusidhu7/uwct-tritonia-v9-inputs`.
3. Cell 1: set `COMMIT` to the pushed commit that contains this notebook and configs E047-E061.
4. Run cells 1-6 and check that cell 6 prints **VERIFIED** twice (v7 and v9).
5. Cell 7: set `RUN_SMOKE = True`; all 5 smoke runs must pass.
6. Cell 8: set `RUN_QUEUE = True` to train the 15 runs, then run cell 9 (comparison) and cell 10 (downloads).
7. **Download both zips before closing the session**: saved versions have kept no output so far.
8. If the session dies mid-run: in a new session, run cells 1-6, then resume an interrupted run with
   `train_yolo.py --resume` (its `last.pt` must still exist).

**Session 2 (D-045 run notes):** E047-E055 are registered at the commit, so cell 7 smoke-tests only E056 and E059,
and cell 8 queues only **E056-E061**. Their parents E047-E052 are registered, which train_yolo.py requires. ESTIMATE
about 40-60 min, from the session-1 resume times (m 4-8 min, l 5-10 min per run on one T4).
- **Run cell 8 once.** Stopping it now also stops its training processes; a second run while a queue is alive is
  refused.
- A run interrupted by a stop keeps its folder and `last.pt`. Finish it with `train_yolo.py --resume`; never re-queue it.
- Cell 9 reads session 1's results from the archived records, so its table covers all five arms.

**Final test -- NOT in this notebook.** Allowed only if cell 9 says an arm **WINS**, and then only after a separate
user approval (D-045 point 6, CLAUDE.md rule 7).

In [ ]:
# 1. Settings and helpers -- set COMMIT; keep everything else.
import os, re, subprocess, sys, json, time, shutil

REPO = "SakshamPainuly/underwater-creature-tracking"
BRANCH = "anu/tritonia-preproc-size"   # branch that holds COMMIT
COMMIT = "0000000000000000000000000000000000000000"  # REQUIRED: full sha, pushed
TOKEN_SECRET_NAME = "UWCT_GITHUB_TOKEN"
# Per dataset version: Kaggle dataset, frames folder name (with images/ and points/), expected view fingerprint.
VIEWS = {
    "v7": dict(kaggle="uwct-tritonia-v7-inputs", batch="tritonia_label_batch1",
               fingerprint="8ea6503cab38c9fed92f1c5be9d71fae5de6af3d43f17a31d533d8bf7854649a"),
    "v9": dict(kaggle="uwct-tritonia-v9-inputs", batch="tritonia_bgc1_v7",
               fingerprint="9bfe2ae5631b4665b90b59f78c0d5531e8486bd3d218ce81428a7d46a5d6f84f"),
}
BOXES_NAME = "tritonia_boxes_fixed2"   # label source of both v7 and v9 (D-045)

WORK = "/tmp"                          # scratch, not saved
REPO_DIR = f"{WORK}/uwct"
LOCAL_BOXES = f"{WORK}/raw/{BOXES_NAME}"
W = "/kaggle/working"
RUNS = f"{W}/runs"                     # run folders (zipped for download in cell 10)
WEIGHTS = f"{WORK}/weights"            # pretrained checkpoints
LOGS = f"{W}/d045_logs"
os.makedirs(RUNS, exist_ok=True); os.makedirs(WEIGHTS, exist_ok=True); os.makedirs(LOGS, exist_ok=True)

assert "KAGGLE_KERNEL_RUN_TYPE" in os.environ, "this notebook is for Kaggle (GPU T4 x2)"
assert re.fullmatch(r"[0-9a-f]{40}", COMMIT) and set(COMMIT) != {"0"}, "Set COMMIT to the exact 40-character commit sha to run"


def sh(*cmd, env=None):
    """Run a command in the repo, stream its output, raise if it fails."""
    p = subprocess.Popen(list(cmd), cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {' '.join(cmd)}")


def get_secret(name):
    from kaggle_secrets import UserSecretsClient
    return UserSecretsClient().get_secret(name)


print("settings OK | runs:", RUNS)

In [ ]:
# 2. GPUs and the attached inputs (read-only under /kaggle/input; exactly one match each, else stop).
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "NO GPU -- switch the accelerator to GPU T4 x2")


def find_input_dir(dataset, name, must_contain):
    hits = sorted(d for d, subdirs, _ in os.walk("/kaggle/input")
                  if os.path.basename(d) == name and must_contain in subdirs and dataset in d)
    assert len(hits) == 1, (f"expected one {name}/ folder (with {must_contain}/) of dataset {dataset} "
                            f"under /kaggle/input, found {hits} -- attach the dataset (Add Input)")
    return hits[0]


SRC_BOXES = find_input_dir(VIEWS["v7"]["kaggle"], BOXES_NAME, "labels")
for v, c in VIEWS.items():
    c["src_batch"] = find_input_dir(c["kaggle"], c["batch"], "images")
    print(v, "| frames:", c["src_batch"])
print("label source:", SRC_BOXES)

In [ ]:
# 3. Check out the exact commit (read-only token from Kaggle Secrets, as in colab_train_tritonia.ipynb cell 4:
#    the token is only in the environment of the one git process, never in a URL, file, .git/config or output).
import stat

REPO_URL = f"https://github.com/{REPO}.git"
ASKPASS = f"{WORK}/uwct_git_askpass.sh"
with open(ASKPASS, "w") as f:
    f.write('#!/bin/sh\n'
            'case "$1" in\n'
            '  Username*) echo "x-access-token" ;;\n'
            '  *) printf "%s\\n" "$UWCT_GIT_TOKEN" ;;\n'
            'esac\n')
os.chmod(ASKPASS, stat.S_IRWXU)


def git(*args, cwd=REPO_DIR):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{r.stderr}")
    return r.stdout.strip()


def git_remote(*args, cwd=REPO_DIR):
    env = {**os.environ, "GIT_ASKPASS": ASKPASS, "GIT_TERMINAL_PROMPT": "0", "UWCT_GIT_TOKEN": get_secret(TOKEN_SECRET_NAME)}
    try:
        r = subprocess.run(["git", "-c", "credential.helper=", *args], cwd=cwd, env=env, capture_output=True, text=True)
        err = r.stderr.replace(env["UWCT_GIT_TOKEN"], "***")
        config = os.path.join(REPO_DIR, ".git", "config")
        leaked = os.path.isfile(config) and env["UWCT_GIT_TOKEN"] in open(config).read()
    finally:
        env.clear()
    if r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{err}")
    assert not leaked, "token found in .git/config"


if not os.path.isdir(REPO_DIR):
    git_remote("clone", "--branch", BRANCH, REPO_URL, REPO_DIR, cwd=WORK)
else:
    git_remote("fetch", "origin", BRANCH)
git("checkout", "--detach", COMMIT)
assert git("rev-parse", "HEAD") == COMMIT
print("commit:", COMMIT, "| working tree:", git("status", "--porcelain") or "clean")
open(f"{LOGS}/code_version.txt", "w").write(f"{REPO_URL}\n{BRANCH}\n{COMMIT}\n")

In [ ]:
# 4. Pinned GPU requirements (ultralytics==8.4.155; torch/CUDA from Kaggle) and the session environment report.
sh(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-gpu.txt")
sh(sys.executable, "-c", "import ultralytics, torch; print('ultralytics', ultralytics.__version__, '| torch', torch.__version__)")
sh(sys.executable, "scripts/env_report.py", "--out", f"{LOGS}/env_report_session.json")

In [ ]:
# 5. Local copies of the label source and both frame batches (nothing is written to /kaggle/input); the label source
#    is a read-only raw root (UWCT_TRITONIA_BOXES_ROOT) for every later script. Then the frozen-registry check of v7 and v9.
if os.path.isdir(LOCAL_BOXES):
    shutil.rmtree(LOCAL_BOXES)
shutil.copytree(SRC_BOXES, LOCAL_BOXES)
for v, c in VIEWS.items():
    c["local_batch"] = f"{WORK}/batch_{v}"
    if os.path.isdir(c["local_batch"]):
        shutil.rmtree(c["local_batch"])
    shutil.copytree(c["src_batch"], c["local_batch"], ignore=shutil.ignore_patterns("hints"))
os.environ["UWCT_TRITONIA_BOXES_ROOT"] = LOCAL_BOXES
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from dataset_audit.versioning import read_version_meta, verify_frozen
for v in VIEWS:
    verify_frozen(v)
    meta = read_version_meta(v)
    assert meta["dataset_kind"] == "tritonia" and meta["status"] == "frozen" and meta["label_source"] == BOXES_NAME, meta
    print(v, "frozen OK |", len([f for f in os.listdir(os.path.join(VIEWS[v]["local_batch"], "images")) if f.endswith(".png")]), "frames")
assert read_version_meta("v9")["derived_from"] == "v7"

In [ ]:
# 6. Build the YOLO views of v7 and v9 (every image and label hash-checked against the frozen version), check their
#    fingerprints, Ultralytics check (must print VERIFIED for each), and check the 15 D-045 configs at this commit.
for v, c in VIEWS.items():
    c["dataset_dir"] = f"{REPO_DIR}/data/yolo/tritonia_{v}"
    sh(sys.executable, "scripts/prepare_tritonia_yolo.py", "--version", v,
       "--frames", os.path.join(c["local_batch"], "images"), "--boxes", LOCAL_BOXES)
    fp = json.load(open(os.path.join(c["dataset_dir"], "build_summary.json")))["dataset_fingerprint"]
    assert fp == c["fingerprint"], (v, fp)
    sh(sys.executable, "scripts/verify_yolo_dataset.py", "--dataset", c["dataset_dir"])

from training.experiment import load_config, validate_config
ARMS = {"M": ["E047", "E048", "E049"], "L": ["E050", "E051", "E052"], "S-bgc": ["E053", "E054", "E055"],
        "M-bgc": ["E056", "E057", "E058"], "L-bgc": ["E059", "E060", "E061"]}
ARM_SPEC = {"M": ("v7", "yolo26m.pt"), "L": ("v7", "yolo26l.pt"), "S-bgc": ("v9", "yolo26s.pt"),
            "M-bgc": ("v9", "yolo26m.pt"), "L-bgc": ("v9", "yolo26l.pt")}
CFG_DIR = os.path.join(REPO_DIR, "configs", "experiments")
CONFIGS = {}
for arm, ids in ARMS.items():
    version, model = ARM_SPEC[arm]
    for seed, e in enumerate(ids):
        (name,) = [f for f in os.listdir(CFG_DIR) if f.startswith(e + "_")]
        cfg = validate_config(load_config(os.path.join(CFG_DIR, name)), name)
        assert (cfg["dataset_version"], cfg["model"], cfg["seed"]) == (version, model, seed), cfg
        assert cfg["dataset_fingerprint"] == VIEWS[version]["fingerprint"], cfg
        CONFIGS[e] = dict(arm=arm, name=name, seed=seed, version=version, model=model,
                          imgsz=cfg["train_args"]["imgsz"], args=cfg["train_args"])
for e, c in CONFIGS.items():
    print(e, c["arm"], c["version"], c["model"], "seed", c["seed"], c["args"])

In [ ]:
# 7. Smoke tests (CLAUDE.md section 6): 1 epoch on 10% of train, validated on val, seed 0 of every arm, two at a
#    time (one per T4). Every training-side check runs (config, clean commit, frozen view, checkpoint).
#    NOT results. Smoke folders are RUNS/<id>_smoke_<utc>/. The times give the first real estimate for m and l.
import threading
RUN_SMOKE = False  # set to True deliberately
assert RUN_SMOKE, "Smoke test not enabled"
N_GPUS = len([l for l in subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines()
              if l.startswith("GPU ")])
LANES = max(1, min(N_GPUS, 2))
ENV = dict(os.environ)  # includes UWCT_TRITONIA_BOXES_ROOT (cell 5)


def train_cmd(e, *extra):
    return [sys.executable, f"{REPO_DIR}/scripts/train_yolo.py", "--config", f"configs/experiments/{CONFIGS[e]['name']}",
            "--runs-root", RUNS, "--weights-dir", WEIGHTS, "--dataset", VIEWS[CONFIGS[e]["version"]]["dataset_dir"], *extra]


PROCS = set()          # every running child process, so stopping cell 8 can stop them too (D-045 run notes)
PROCS_LOCK = threading.Lock()


def run_on_gpu(cmd, gpu, log_path):
    """Run one training / eval process on one GPU. Never overwrites a log: an existing one gets a numbered name."""
    base, k = log_path, 1
    while os.path.exists(log_path):
        log_path = f"{base[:-4]}.{k}.log"; k += 1
    with open(log_path, "w") as log:
        p = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT, text=True,
                             env=dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu)), start_new_session=True)
        with PROCS_LOCK:
            PROCS.add(p)
        try:
            return p.wait()
        finally:
            with PROCS_LOCK:
                PROCS.discard(p)


def stop_all_processes():
    """Terminate every child process started by run_on_gpu (and its process group), then wait for them."""
    import signal
    with PROCS_LOCK:
        procs = list(PROCS)
    for p in procs:
        try:
            os.killpg(p.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
    for p in procs:
        try:
            p.wait(timeout=60)
        except subprocess.TimeoutExpired:
            os.killpg(p.pid, signal.SIGKILL)
    return len(procs)


# Runs registered at this commit are finished and reviewed (session 1: E047-E055); only the rest are smoke-tested
# and queued. Their parents must be registered too (train_yolo.py checks it).
import csv as _csv
REGISTERED = {r["experiment_id"] for r in _csv.DictReader(open(f"{REPO_DIR}/experiments/registry.csv", encoding="utf-8"))}
TO_RUN = [e for e in CONFIGS if e not in REGISTERED]
print("already registered (skipped):", sorted(set(CONFIGS) - set(TO_RUN)), "| to run:", TO_RUN)
smoke_ids = [e for e in ["E059", "E050", "E056", "E047", "E053"] if e in TO_RUN]
smoke = {}
def smoke_lane(gpu, ids):
    for e in ids:
        t0 = time.time()
        rc = run_on_gpu(train_cmd(e, "--smoke"), gpu, f"{LOGS}/smoke_{e}.log")
        smoke[e] = dict(exit=rc, seconds=round(time.time() - t0))
        print(f"smoke {e} (arm {CONFIGS[e]['arm']}) GPU {gpu}: exit {rc}, {smoke[e]['seconds']} s", flush=True)
threads = [threading.Thread(target=smoke_lane, args=(g, smoke_ids[g::LANES]), daemon=True) for g in range(LANES)]
[t.start() for t in threads]
try:
    while any(t.is_alive() for t in threads):
        [t.join(timeout=5) for t in threads]
except KeyboardInterrupt:
    print("STOPPED by user: terminated", stop_all_processes(), "smoke process(es)")
    raise
json.dump(smoke, open(f"{LOGS}/smoke_summary.json", "w"), indent=1)
bad = [e for e, s in smoke.items() if s["exit"] != 0]
for e in bad:
    print(f"--- {e} log tail:\n" + "\n".join(open(f"{LOGS}/smoke_{e}.log", errors="replace").read().splitlines()[-15:]))
assert not bad, f"smoke failed: {bad}"
print(f"ALL {len(smoke)} SMOKE RUNS PASSED")

In [ ]:
# 8. D-045: train E047-E061 on both GPUs (one process per T4), seed 0 of every arm first, largest model first.
#    After each run: val point eval of its best.pt on ITS OWN images (v7 runs on raw frames, v9 runs on bgc1 frames;
#    all 27 val frames, threshold = its best val F1, at its imgsz). Val only; the test split is never touched.
#    Status every 5 min; LOGS/queue_status.json is updated as it goes.
#    Stopping this cell stops its training processes too; running it again while a queue is alive is refused.
#    Runs registered at this commit are skipped (cell 7: TO_RUN); a run whose folder already exists is not retrained
#    here -- finish it with train_yolo.py --resume instead.
RUN_QUEUE = False  # set to True deliberately, after cell 7 passed
assert RUN_QUEUE, "Queue not enabled"
assert not any(t.is_alive() for t in globals().get("QUEUE_THREADS", [])),     "a previous cell-8 queue is still running in this session; wait for it (or restart the session)"
FULL_QUEUE = ["E059", "E050", "E056", "E047", "E053",
              "E060", "E051", "E057", "E048", "E054",
              "E061", "E052", "E058", "E049", "E055"]
assert sorted(FULL_QUEUE) == sorted(CONFIGS), "queue and configs differ"
QUEUE = [e for e in FULL_QUEUE if e in TO_RUN]
existing = [e for e in QUEUE if os.path.exists(f"{RUNS}/{e}")]
assert not existing, f"run folders already exist for {existing}: finish them with --resume, do not re-queue them"
pending = list(QUEUE)
status = {e: {"arm": CONFIGS[e]["arm"], "seed": CONFIGS[e]["seed"], "state": "queued"} for e in QUEUE}
LOCK = threading.Lock()
T_QUEUE = time.time()


def save_status():
    json.dump({"elapsed_min": round((time.time() - T_QUEUE) / 60, 1), "runs": status},
              open(f"{LOGS}/queue_status.json", "w"), indent=1)


def lane(gpu):
    while True:
        with LOCK:
            if not pending:
                return
            e = pending.pop(0)
            status[e].update(state="training", gpu=gpu)
            save_status()
        print(f"START {e} (arm {CONFIGS[e]['arm']}, seed {CONFIGS[e]['seed']}) on GPU {gpu}", flush=True)
        t0 = time.time()
        rc = run_on_gpu(train_cmd(e), gpu, f"{LOGS}/train_{e}.log")
        rec_path = f"{RUNS}/{e}/run_record.json"
        if rc != 0 or not os.path.exists(rec_path):
            tail = open(f"{LOGS}/train_{e}.log", errors="replace").read().splitlines()[-15:]
            with LOCK:
                status[e].update(state=f"failed training (exit {rc})", minutes=round((time.time() - t0) / 60, 1), log_tail=tail)
                save_status()
            print(f"FAILED {e} exit {rc}:\n  " + "\n  ".join(tail), flush=True)
            continue
        rec = json.load(open(rec_path))
        cmd = [sys.executable, f"{REPO_DIR}/scripts/tritonia_point_eval.py",
               "--batch", VIEWS[CONFIGS[e]["version"]]["local_batch"], "--split", "val",
               "--weights", rec["outputs"]["best"]["path"], "--imgsz", str(CONFIGS[e]["imgsz"]),
               "--pred-dir", f"{RUNS}/{e}/val_point_predictions", "--out", f"{RUNS}/{e}/val_point_scores.json"]
        rc2 = run_on_gpu(cmd, gpu, f"{LOGS}/pointeval_{e}.log")
        with LOCK:
            if rc2 == 0:
                v = json.load(open(f"{RUNS}/{e}/val_point_scores.json"))
                status[e].update(state="finished", minutes=round((time.time() - t0) / 60, 1),
                                 epochs=rec.get("epochs_completed"), point_ap=v["point_ap"], at_conf=v["at_conf"],
                                 per_day=v["per_day"], ultralytics_val=rec["val_metrics"],
                                 best_sha256=rec["outputs"]["best"]["sha256"])
            else:
                status[e].update(state=f"trained; point eval failed (exit {rc2})")
            save_status()
        if rc2 == 0:
            a = v["at_conf"]
            print(f"DONE  {e} arm {CONFIGS[e]['arm']} seed {CONFIGS[e]['seed']}: {status[e]['minutes']} min, "
                  f"{rec.get('epochs_completed')} epochs | VAL point AP {v['point_ap']:.3f} | P/R/F1 {a['precision']:.3f}/"
                  f"{a['recall']:.3f}/{a['f1']:.3f} at conf {a['conf']}", flush=True)
        else:
            print(f"POINT EVAL FAILED {e}: see {LOGS}/pointeval_{e}.log", flush=True)


print(f"{LANES} lane(s); {len(QUEUE)} runs; {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())}")
QUEUE_THREADS = [threading.Thread(target=lane, args=(g,), daemon=True) for g in range(LANES)]
[t.start() for t in QUEUE_THREADS]
try:
    while any(t.is_alive() for t in QUEUE_THREADS):
        for t in QUEUE_THREADS:
            t.join(timeout=150)
        with LOCK:
            busy = [e for e, s in status.items() if s["state"] == "training"]
            done = sum(s["state"] == "finished" for s in status.values())
        print(f"  .. {round((time.time() - T_QUEUE) / 60)} min: {done}/{len(QUEUE)} finished; training {busy}", flush=True)
except KeyboardInterrupt:
    with LOCK:
        stopped_queued = list(pending); pending.clear()   # lanes take nothing new
    n = stop_all_processes()
    for t in QUEUE_THREADS:
        t.join(timeout=60)
    with LOCK:
        for e in stopped_queued:
            status[e]["state"] = "not started (queue stopped by user)"
        save_status()
    print(f"STOPPED by user: terminated {n} process(es); never started: {stopped_queued}. A run that was training "
          "keeps its folder and last.pt: finish it with train_yolo.py --resume, do not re-run this cell for it.")
    raise
save_status()
print("\nQUEUE FINISHED")
for e, s in status.items():
    print(f"  {e} arm {s['arm']} seed {s['seed']}: {s['state']}" + (f"  point AP {s['point_ap']:.3f}" if "point_ap" in s else ""))

In [ ]:
# 9. D-045 comparison (val only): per arm the mean val point AP over its 3 seeds, the seed range and the declared rule,
#    plus the images x size table (descriptive). Baseline = E010 / E042 / E043, registered, not re-run: E042 / E043 from
#    their archived val_point_scores.json at this commit, E010 from docs/EXPERIMENTS.md (0.496 at conf 0.16; its val file
#    is not archived). Writes RUNS/d045_summary.json and a chart. Claude recomputes this from the files before any decision.
import statistics
#    Runs registered at this commit (session 1) come from their archived experiments/records/<id>/val_point_scores.json;
#    the others from this session's queue_status.json.
st = json.load(open(f"{LOGS}/queue_status.json"))["runs"] if os.path.exists(f"{LOGS}/queue_status.json") else {}
for e in CONFIGS:
    arch = f"{REPO_DIR}/experiments/records/{e}/val_point_scores.json"
    if e in REGISTERED and os.path.exists(arch):
        a = json.load(open(arch))
        st[e] = {"state": "finished", "point_ap": a["point_ap"], "at_conf": a["at_conf"], "per_day": a["per_day"],
                 "source": "archived record"}
base_runs = [("E010", 0.496)] + [(e, json.load(open(f"{REPO_DIR}/experiments/records/{e}/val_point_scores.json"))["point_ap"])
                                 for e in ("E042", "E043")]
base = [a for _, a in base_runs]
b_mean, b_range = statistics.mean(base), max(base) - min(base)
rows = ["| arm | images | model | runs | val point AP mean | seed range | per seed | mean F1 | verdict |",
        "|---|---|---|---:|---:|---:|---|---:|---|",
        f"| baseline | v7 | yolo26s | 3 | {b_mean:.3f} | {b_range:.3f} | {', '.join(f'{e} {a:.3f}' for e, a in base_runs)} | - | reference |"]
summary, means = {}, {("v7", "yolo26s.pt"): b_mean}
for arm, ids in ARMS.items():
    vals = [(e, st[e]) for e in ids if st.get(e, {}).get("state") == "finished"]
    version, model = ARM_SPEC[arm]
    if not vals:
        rows.append(f"| {arm} | {version} | {model[:-3]} | 0 | - | - | - | - | no finished runs |"); continue
    aps = [v["point_ap"] for _, v in vals]
    mean, rng = statistics.mean(aps), max(aps) - min(aps)
    f1 = statistics.mean(v["at_conf"]["f1"] for _, v in vals)
    if len(aps) < 3:
        verdict = "cannot win (fewer than 3 seeds)"
    else:
        verdict = "WINS" if mean - b_mean > max(rng, b_range) else "no evidence of a gain"
    means[(version, model)] = mean
    summary[arm] = dict(runs=[e for e, _ in vals], point_ap=aps, mean=round(mean, 4), range=round(rng, 4),
                        mean_f1=round(f1, 4), verdict=verdict,
                        per_day_f1={d: round(statistics.mean(v["per_day"][d]["f1"] for _, v in vals), 4)
                                    for d in sorted(vals[0][1]["per_day"])})
    rows.append(f"| {arm} | {version} | {model[:-3]} | {len(aps)} | {mean:.3f} | {rng:.3f} | "
                f"{', '.join(f'{e} {a:.3f}' for (e, _), a in zip(vals, aps))} | {f1:.3f} | {verdict} |")
grid = ["| images / model | yolo26s | yolo26m | yolo26l |", "|---|---:|---:|---:|"]
for version, label in (("v7", "raw (v7)"), ("v9", "bgc1 (v9)")):
    grid.append(f"| {label} | " + " | ".join(f"{means[(version, m)]:.3f}" if (version, m) in means else "-"
                                             for m in ("yolo26s.pt", "yolo26m.pt", "yolo26l.pt")) + " |")
from IPython.display import Markdown, display
display(Markdown("## D-045 -- val point AP (not mAP; vs the MTrackJ crosses)\n" + "\n".join(rows) +
                 f"\n\nRule: an arm wins if its mean exceeds the baseline mean ({b_mean:.3f}) by more than the larger of "
                 f"its own and the baseline's seed range ({b_range:.3f}).\n\n### Mean val point AP, images x size "
                 "(descriptive only)\n" + "\n".join(grid)))
json.dump({"baseline_runs": base_runs, "baseline_mean": b_mean, "baseline_range": b_range, "arms": summary},
          open(f"{RUNS}/d045_summary.json", "w"), indent=1)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
xs = {"yolo26s.pt": 0, "yolo26m.pt": 1, "yolo26l.pt": 2}
for version, colour, dx in (("v7", "#888888", -0.18), ("v9", "#2a9d8f", 0.18)):
    for arm, (v, m) in ARM_SPEC.items():
        if v == version and arm in summary:
            ax.bar(xs[m] + dx, summary[arm]["mean"], width=0.34, color=colour, alpha=0.85)
            ax.scatter([xs[m] + dx] * len(summary[arm]["point_ap"]), summary[arm]["point_ap"], color="k", s=12, zorder=3)
    if version == "v7":
        ax.bar(xs["yolo26s.pt"] + dx, b_mean, width=0.34, color=colour, alpha=0.85, label="raw (v7)")
        ax.scatter([xs["yolo26s.pt"] + dx] * 3, base, color="k", s=12, zorder=3)
ax.bar([], [], color="#2a9d8f", label="bgc1 (v9)")
ax.axhline(b_mean, color="#888888", lw=0.8, ls="--")
ax.set_xticks([0, 1, 2], ["yolo26s", "yolo26m", "yolo26l"])
ax.set_ylabel("val point AP (bar = mean; dots = seeds)")
ax.set_title("Tritonia D-045, 27 val frames")
ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(f"{RUNS}/d045_val_point_ap.png", dpi=150)
display(fig)

In [ ]:
# 10. Download links (saved versions have kept no output so far, so download now):
#     d045_results.zip  = run records, val point scores, results.csv, args.yaml, logs, summary + chart (small)
#     d045_weights.zip  = best.pt of every finished run (about 20-100 MB each)
import zipfile
from IPython.display import FileLink
os.chdir(W)
with zipfile.ZipFile("d045_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for d, _, files in os.walk("runs"):
        for n in sorted(files):
            p = os.path.join(d, n)
            if n.endswith(".pt") or n.endswith(".jpg") or "_smoke_" in p:
                continue
            z.write(p)
    for d, _, files in os.walk("d045_logs"):
        for n in sorted(files):
            z.write(os.path.join(d, n))
with zipfile.ZipFile("d045_weights.zip", "w", zipfile.ZIP_STORED) as z:
    for e in sorted(CONFIGS):
        p = f"runs/{e}/train/weights/best.pt"
        if os.path.exists(p):
            z.write(p)
for f in ("d045_results.zip", "d045_weights.zip"):
    print(f, os.path.getsize(f), "bytes")
    display(FileLink(f))